# 基于 TD3 的 PMM 策略强化学习训练

本 notebook 实现了基于 **TD3 (Twin Delayed Deep Deterministic Policy Gradient)** 算法的 PMM 做市策略训练系统。

## TD3 算法特点

- **确定性策略**: 输出确定性动作，适合连续控制任务
- **双Q网络**: 使用两个Critic网络减少Q值过估计
- **延迟策略更新**: 减少策略更新频率，提高稳定性
- **目标策略平滑**: 在目标动作中添加噪声，提高鲁棒性
- **裁剪双Q学习**: 使用较小的Q值作为目标，减少过估计

## 与 SAC 的对比

| 特性 | TD3 | SAC |
|------|-----|-----|
| 策略类型 | 确定性 | 随机性 |
| 探索方式 | 添加噪声 | 熵正则化 |
| 更新策略 | 延迟更新 | 每步更新 |
| 目标平滑 | ✅ | ❌ |
| 计算效率 | 更快 | 稍慢 |

## 🚀 快速开始

### 训练配置（重要！）

在运行训练前，请先在 **第一个代码Cell** 中设置训练参数：
```python
NUM_EPISODES = 20    # 直接设置训练轮数：20(测试), 100(快速), 500(标准), 1000(深度)
BATCH_SIZE = 128     # 批量大小（内存不足时减小）
```

### 执行顺序

1. **Cell 1**: 训练配置（设置NUM_EPISODES等参数）
2. **Cell 2**: 环境设置和设备选择
3. **Cell 3**: 数据配置和切片准备
4. **Cell 4**: TD3 神经网络组件
5. **Cell 5**: TD3 算法实现
6. **Cell 6**: 训练配置（自动使用全局参数）
7. **Cell 7**: 环境管理器
8. **Cell 8**: 训练循环
9. **Cell 9**: 执行训练
10. **Cell 10**: 模型评估

### ⚠️ 注意事项

- **快速测试**: 设置 `NUM_EPISODES=20` 进行5-10分钟测试
- **标准训练**: 设置 `NUM_EPISODES=500` 获得稳定策略
- **内存不足**: 减少 `BATCH_SIZE` 和 `REPLAY_BUFFER_SIZE`
- **保存进度**: 训练会定期保存 checkpoint，可恢复

In [21]:
# ⚠️ 训练配置（在运行其他代码前先设置此项）
# ====================================================

# 直接设置训练轮数（推荐方式）
NUM_EPISODES = 200         # 可自由设置: 20(测试), 100(快速), 500(标准), 1000(深度)

# 其他可调参数
BATCH_SIZE = 128          # 批量大小（内存不足时可减小到64）
REPLAY_BUFFER_SIZE = 20000  # 经验池大小（内存不足时可减小到10000）
DATA_SAMPLE_RATE = 0.1    # 数据采样率（0.05-0.2）
SAVE_INTERVAL = 50        # 模型保存间隔

# 显示当前配置
print("=" * 50)
print("📋 TD3训练配置")
print("=" * 50)

# 根据训练轮数判断模式
if NUM_EPISODES <= 20:
    print("🧪 模式: 快速测试")
    estimated_time = "5-10分钟"
elif NUM_EPISODES <= 100:
    print("⚡ 模式: 快速训练")
    estimated_time = "30-60分钟"
elif NUM_EPISODES <= 500:
    print("💪 模式: 标准训练")
    estimated_time = "4-6小时"
else:
    print("🔥 模式: 深度训练")
    estimated_time = "8-12小时"

print(f"📊 训练轮数: {NUM_EPISODES:,} episodes")
print(f"💾 批量大小: {BATCH_SIZE}")
print(f"🗄️ 经验池大小: {REPLAY_BUFFER_SIZE:,}")
print(f"📉 数据采样率: {DATA_SAMPLE_RATE*100:.0f}%")
print(f"💾 保存间隔: 每{SAVE_INTERVAL}轮")
print(f"\n⏱️ 预计时间: {estimated_time}")

print("\n💡 建议配置:")
print("   • 测试环境: NUM_EPISODES=20")
print("   • 快速原型: NUM_EPISODES=100")
print("   • 标准训练: NUM_EPISODES=500")
print("   • 最佳效果: NUM_EPISODES=1000")
print("=" * 50)

📋 TD3训练配置
💪 模式: 标准训练
📊 训练轮数: 200 episodes
💾 批量大小: 128
🗄️ 经验池大小: 20,000
📉 数据采样率: 10%
💾 保存间隔: 每50轮

⏱️ 预计时间: 4-6小时

💡 建议配置:
   • 测试环境: NUM_EPISODES=20
   • 快速原型: NUM_EPISODES=100
   • 标准训练: NUM_EPISODES=500
   • 最佳效果: NUM_EPISODES=1000


In [22]:
# Cell 1: 环境设置和依赖导入
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warnings
import os
from collections import deque
import random
from tensordict import TensorDict
from hftbacktest import BacktestAsset
from lib.rl_env import create_pmm_env
from lib.data_slicer import DataSlicer
from tqdm.notebook import tqdm
import json
import time
import copy

# 设置警告过滤和随机种子
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 🔧 全局交易配置
MAKER_FEE_RATE = -0.00003     # Maker费率 -0.003% (负费率返佣)
TAKER_FEE_RATE = 0.0007       # Taker费率 +0.07% (正费率收费)
TICK_SIZE = 0.0001            # XRP最小价格变动单位
LOT_SIZE = 0.1                # XRP最小交易数量单位
TRAINING_PAIR = 'xrpusdt'     # XRP交易对
START_DATE = 20250717         # 使用更新的数据

print(f"📊 全局交易配置:")
print(f"   交易对: {TRAINING_PAIR.upper()}")
print(f"   Maker费率: {MAKER_FEE_RATE*100:.4f}% (负费率返佣)")
print(f"   Taker费率: {TAKER_FEE_RATE*100:.4f}% (正费率收费)")
print(f"   最小价格单位: {TICK_SIZE}")
print(f"   最小交易单位: {LOT_SIZE}")

# 🔧 设备配置（智能选择：CUDA > MPS > CPU）
def select_device():
    """智能选择最佳可用设备"""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("🚀 使用 CUDA GPU 加速")
        print(f"   GPU型号: {torch.cuda.get_device_name(0)}")
        print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        torch.cuda.empty_cache()
        return device
    
    if torch.backends.mps.is_available():
        try:
            test_tensor = torch.tensor([1.0], device="mps")
            _ = test_tensor * 2
            device = torch.device("mps")
            print("🍎 使用 Apple Silicon MPS 加速")
            print("   提示：MPS 加速可能显著提升训练速度")
            return device
        except Exception as e:
            print(f"⚠️ MPS 可用但初始化失败: {e}")
            print("   回退到 CPU 模式")
    
    device = torch.device("cpu")
    print("💻 使用 CPU 运行")
    print("   提示：训练速度较慢，建议使用 GPU 或 MPS")
    import platform
    print(f"   处理器: {platform.processor()}")
    print(f"   CPU核心数: {os.cpu_count()}")
    return device

device = select_device()
print(f"✅ TD3强化学习环境初始化完成，使用设备: {device}")

📊 全局交易配置:
   交易对: XRPUSDT
   Maker费率: -0.0030% (负费率返佣)
   Taker费率: 0.0700% (正费率收费)
   最小价格单位: 0.0001
   最小交易单位: 0.1
🍎 使用 Apple Silicon MPS 加速
   提示：MPS 加速可能显著提升训练速度
✅ TD3强化学习环境初始化完成，使用设备: mps


In [23]:
# Cell 2: 数据配置
from lib.data_slicer import DataSlicer
print("📊 初始化数据系统...")

# 检查数据文件
data_file = f'data/output/{TRAINING_PAIR}_{START_DATE}.npz'
if not os.path.exists(data_file):
    raise FileNotFoundError(f"数据文件不存在: {data_file}")

print(f"✅ 数据文件: {os.path.basename(data_file)}")

# 创建数据片段
print("📊 准备数据片段（按时间切片）...")
slicer = DataSlicer(slices_dir='data/slices')
data_splits = slicer.split_data_by_time(
    data_file=data_file,
    pair_name=TRAINING_PAIR,
    start_date=START_DATE,
    hours_per_split=0.167  # 每10分钟一个片段（10/60小时）
)
print(f"✅ 获得 {len(data_splits)} 个数据片段")

data_files = data_splits
print(f"✅ 数据准备完成，共 {len(data_files)} 个训练片段")

# 显示前几个切片的信息
for i, split_file in enumerate(data_files[:3]):
    info = slicer.get_split_info(split_file)
    print(f"   片段{i}: {info['records']:,} 条记录, {info['duration_hours']:.1f}小时")

📊 初始化数据系统...
✅ 数据文件: xrpusdt_20250717.npz
📊 准备数据片段（按时间切片）...
📦 发现已存在的数据分片 (144个)，直接使用...
   片段0: 349,899 条记录 (0.3M) - 0.2小时
   片段1: 330,220 条记录 (0.3M) - 0.2小时
   片段2: 343,814 条记录 (0.3M) - 0.2小时
   ... 共144个片段
✅ 获得 144 个数据片段
✅ 数据准备完成，共 144 个训练片段
   片段0: 349,899 条记录, 0.2小时
   片段1: 330,220 条记录, 0.2小时
   片段2: 343,814 条记录, 0.2小时


In [24]:
# Cell 3: TD3 神经网络组件

class Actor(nn.Module):
    """TD3 Actor网络 - 确定性策略"""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256, max_action=1.0):
        super(Actor, self).__init__()
        self.max_action = max_action
        
        # 网络层
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
        # 初始化权重
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        action = torch.tanh(self.fc3(x))
        return action * self.max_action


class Critic(nn.Module):
    """TD3 Critic网络 - 双Q网络架构"""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(Critic, self).__init__()
        
        # Q1 网络
        self.q1_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q1_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q1_out = nn.Linear(hidden_dim, 1)
        
        # Q2 网络
        self.q2_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q2_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q2_out = nn.Linear(hidden_dim, 1)
        
        # 初始化权重
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()
    
    def forward(self, state, action):
        xu = torch.cat([state, action], dim=1)
        
        # Q1 前向传播
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)
        
        # Q2 前向传播
        q2 = F.relu(self.q2_fc1(xu))
        q2 = F.relu(self.q2_fc2(q2))
        q2 = self.q2_out(q2)
        
        return q1, q2
    
    def Q1(self, state, action):
        """只计算Q1值"""
        xu = torch.cat([state, action], dim=1)
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)
        return q1


class ReplayBuffer:
    """经验回放缓冲区"""
    
    def __init__(self, capacity=1000000):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """添加经验到缓冲区"""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """随机采样一批经验"""
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    
    def __len__(self):
        return len(self.buffer)


print("✅ TD3神经网络组件定义完成")
print(f"   - Actor网络: 确定性策略网络")
print(f"   - Critic网络: 双Q网络架构")
print(f"   - ReplayBuffer: 经验回放缓冲区")

✅ TD3神经网络组件定义完成
   - Actor网络: 确定性策略网络
   - Critic网络: 双Q网络架构
   - ReplayBuffer: 经验回放缓冲区


In [25]:
# Cell 4: TD3 算法实现

class TD3:
    """Twin Delayed Deep Deterministic Policy Gradient算法实现"""
    
    def __init__(
        self,
        state_dim,
        action_dim,
        action_low,
        action_high,
        device,
        lr_actor=3e-4,
        lr_critic=3e-4,
        gamma=0.99,
        tau=0.005,
        policy_noise=0.2,
        noise_clip=0.5,
        policy_freq=2,
        exploration_noise=0.1
    ):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_freq = policy_freq
        self.exploration_noise = exploration_noise
        
        # 动作空间边界
        self.action_low = torch.tensor(action_low, device=device)
        self.action_high = torch.tensor(action_high, device=device)
        self.action_scale = (self.action_high - self.action_low) / 2.0
        self.action_bias = (self.action_high + self.action_low) / 2.0
        
        # 创建网络
        self.actor = Actor(state_dim, action_dim).to(device)
        self.actor_target = Actor(state_dim, action_dim).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        
        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = Critic(state_dim, action_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        
        # 优化器
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # 计数器
        self.total_it = 0
    
    def select_action(self, state, add_noise=True):
        """选择动作"""
        state = torch.FloatTensor(state).to(self.device).unsqueeze(0)
        
        with torch.no_grad():
            action = self.actor(state).squeeze(0).cpu().numpy()
        
        # 添加探索噪声
        if add_noise:
            noise = np.random.normal(0, self.exploration_noise, size=action.shape)
            action = action + noise
        
        # 将动作从[-1, 1]缩放到实际范围
        action = action * self.action_scale.cpu().numpy() + self.action_bias.cpu().numpy()
        
        # 确保动作在边界内并且是整数（对于离散参数）
        action = np.clip(action, self.action_low.cpu().numpy(), self.action_high.cpu().numpy())
        action[0] = round(action[0])  # half_spread
        action[1] = round(action[1])  # skew
        action[2] = round(action[2])  # grid_num
        action[3] = round(action[3])  # grid_interval
        
        return action
    
    def update(self, replay_buffer, batch_size=256):
        """更新网络参数"""
        self.total_it += 1
        
        if len(replay_buffer) < batch_size:
            return {}
        
        # 从经验池采样
        state, action, reward, next_state, done = replay_buffer.sample(batch_size)
        
        state = torch.FloatTensor(state).to(self.device)
        next_state = torch.FloatTensor(next_state).to(self.device)
        action = torch.FloatTensor(action).to(self.device)
        reward = torch.FloatTensor(reward).to(self.device).unsqueeze(1)
        done = torch.FloatTensor(done).to(self.device).unsqueeze(1)
        
        # 将动作归一化到[-1, 1]
        action_normalized = (action - self.action_bias) / self.action_scale
        
        with torch.no_grad():
            # 选择下一个动作并添加噪声（目标策略平滑）
            next_action = self.actor_target(next_state)
            noise = torch.randn_like(next_action) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_action = (next_action + noise).clamp(-1, 1)
            
            # 计算目标Q值
            target_q1, target_q2 = self.critic_target(next_state, next_action)
            target_q = torch.min(target_q1, target_q2)
            target_q_value = reward + (1 - done) * self.gamma * target_q
        
        # 更新Critic
        current_q1, current_q2 = self.critic(state, action_normalized)
        critic_loss = F.mse_loss(current_q1, target_q_value) + F.mse_loss(current_q2, target_q_value)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # 延迟策略更新
        actor_loss = None
        if self.total_it % self.policy_freq == 0:
            # 计算Actor损失
            actor_loss = -self.critic.Q1(state, self.actor(state)).mean()
            
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            # 软更新目标网络
            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
            
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        
        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item() if actor_loss is not None else 0,
            'total_iterations': self.total_it
        }
    
    def save(self, filepath):
        """保存模型"""
        torch.save({
            'actor_state_dict': self.actor.state_dict(),
            'actor_target_state_dict': self.actor_target.state_dict(),
            'critic_state_dict': self.critic.state_dict(),
            'critic_target_state_dict': self.critic_target.state_dict(),
            'actor_optimizer_state_dict': self.actor_optimizer.state_dict(),
            'critic_optimizer_state_dict': self.critic_optimizer.state_dict(),
            'total_it': self.total_it
        }, filepath)
    
    def load(self, filepath):
        """加载模型"""
        checkpoint = torch.load(filepath, map_location=self.device)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.actor_target.load_state_dict(checkpoint['actor_target_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.critic_target.load_state_dict(checkpoint['critic_target_state_dict'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer_state_dict'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer_state_dict'])
        self.total_it = checkpoint['total_it']


print("✅ TD3算法实现完成")
print(f"   - 确定性策略: 无熵正则化")
print(f"   - 延迟更新: 每{2}步更新一次策略")
print(f"   - 目标平滑: 减少目标Q值方差")
print(f"   - 双Q网络: 减少Q值过估计")

✅ TD3算法实现完成
   - 确定性策略: 无熵正则化
   - 延迟更新: 每2步更新一次策略
   - 目标平滑: 减少目标Q值方差
   - 双Q网络: 减少Q值过估计


In [26]:
# Cell 5: 训练配置

TD3_CONFIG = {
    # 环境参数
    'state_dim': 4,                    # 观测维度
    'action_dim': 4,                   # 动作维度
    'action_low': [1.0, 1.0, 5.0, 1.0],    # 动作下界
    'action_high': [20.0, 30.0, 10.0, 20.0],  # 动作上界
    
    # TD3超参数
    'lr_actor': 3e-4,                  # Actor学习率
    'lr_critic': 3e-4,                 # Critic学习率
    'gamma': 0.99,                     # 折扣因子
    'tau': 0.005,                      # 软更新系数
    'policy_noise': 0.2,               # 目标策略噪声
    'noise_clip': 0.5,                 # 噪声裁剪
    'policy_freq': 2,                  # 策略更新频率
    'exploration_noise': 0.1,          # 探索噪声
    
    # 训练参数（使用全局配置）
    'batch_size': BATCH_SIZE,                 # 批量大小
    'replay_buffer_size': REPLAY_BUFFER_SIZE, # 经验池大小
    'num_episodes': NUM_EPISODES,             # 总训练轮数
    'start_steps': 500,                      # 随机探索步数
    'update_interval': 1,                     # 更新间隔
    'eval_interval': 50,                      # 评估间隔
    'save_interval': SAVE_INTERVAL,           # 保存间隔
    
    # 环境参数
    'step_interval_ns': 2_500_000_000,  # 步间隔（1秒）
    'max_steps_per_episode': 500,       # 每轮最大步数
    
    # 数据管理（使用全局配置）
    'use_data_slices': True,                  # 使用数据切片
    'hours_per_slice': 0.167,                 # 每切片小时数（10分钟）
    'slices_per_episode': 1,                  # 每轮使用切片数
    'data_sample_rate': DATA_SAMPLE_RATE,     # 数据采样率
    'max_samples_per_env': 5000000,           # 最大样本数限制
}

# 创建模型保存目录
os.makedirs('checkpoints/td3', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("✅ TD3训练配置完成（使用全局配置）")
print(f"   状态维度: {TD3_CONFIG['state_dim']}")
print(f"   动作维度: {TD3_CONFIG['action_dim']}")
print(f"   学习率: Actor={TD3_CONFIG['lr_actor']}, Critic={TD3_CONFIG['lr_critic']}")
print(f"   批量大小: {TD3_CONFIG['batch_size']} (全局配置)")
print(f"   经验池容量: {TD3_CONFIG['replay_buffer_size']:,} (全局配置)")
print(f"   训练轮数: {TD3_CONFIG['num_episodes']:,} (全局配置)")
print(f"   策略更新频率: 每{TD3_CONFIG['policy_freq']}步")
print(f"   探索噪声: {TD3_CONFIG['exploration_noise']}")

✅ TD3训练配置完成（使用全局配置）
   状态维度: 4
   动作维度: 4
   学习率: Actor=0.0003, Critic=0.0003
   批量大小: 128 (全局配置)
   经验池容量: 20,000 (全局配置)
   训练轮数: 200 (全局配置)
   策略更新频率: 每2步
   探索噪声: 0.1


In [27]:
# Cell 6: 环境管理器

class EnvironmentManager:
    """管理多个数据切片的训练环境"""
    
    def __init__(self, data_files, config, device):
        self.data_files = data_files
        self.config = config
        self.device = device
        self.current_idx = 0
        self.slicer = DataSlicer()
        self.sample_rate = config.get('data_sample_rate', 0.1)
        self.max_samples = config.get('max_samples_per_env', 500000)
        print(f"   环境管理器使用设备: {self.device}")
        print(f"   数据采样率: {self.sample_rate*100:.0f}%")
        print(f"   最大样本数: {self.max_samples:,}")
    
    def create_env_from_slice(self, data_file, sample_rate=None):
        """从数据切片创建环境"""
        import gc
        
        if sample_rate is None:
            sample_rate = self.sample_rate
        
        # 清理内存
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
        elif self.device.type == 'mps':
            torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
        
        # 加载数据（静默模式）
        data = np.load(data_file)
        data_array = data['data']
        
        original_size = len(data_array)
        
        # 计算采样数量
        target_sample_size = int(original_size * sample_rate)
        actual_sample_size = min(target_sample_size, self.max_samples)
        
        # 采样数据
        if actual_sample_size < original_size:
            indices = np.sort(np.random.choice(
                original_size, actual_sample_size, replace=False))
            data_array = data_array[indices]
        
        data.close()
        del data
        gc.collect()
        
        # 创建HFT回测资产
        data_asset = (
            BacktestAsset()
            .data([data_array])
            .linear_asset(1.0)
            .risk_adverse_queue_model()
            .no_partial_fill_exchange()
            .constant_latency(10_000_000, 10_000_000)
            .tick_size(TICK_SIZE)
            .lot_size(LOT_SIZE)
            .trading_value_fee_model(MAKER_FEE_RATE, TAKER_FEE_RATE)
            .power_prob_queue_model(3.0)
        )
        
        # 创建PMM环境
        env = create_pmm_env(
            data_asset=data_asset,
            action_low=self.config['action_low'],
            action_high=self.config['action_high'],
            max_steps=self.config['max_steps_per_episode'],
            device=str(self.device),
            risk_penalty_weight=0.01,
            step_interval_ns=self.config['step_interval_ns'],
        )
        
        return env
    
    def get_next_env(self):
        """获取下一个训练环境"""
        data_file = self.data_files[self.current_idx]
        env = self.create_env_from_slice(data_file)
        self.current_idx = (self.current_idx + 1) % len(self.data_files)
        return env, data_file
    
    def get_random_env(self):
        """随机获取一个训练环境"""
        data_file = random.choice(self.data_files)
        self.current_idx = self.data_files.index(data_file)
        env = self.create_env_from_slice(data_file)
        return env, data_file


# 创建环境管理器
data_slices = data_files
print(f"✅ 使用 {len(data_slices)} 个数据切片")
print(f"   每切片时长: {TD3_CONFIG['hours_per_slice']}小时")
print(f"   数据采样率: {TD3_CONFIG.get('data_sample_rate', 0.1)*100:.0f}%")
print(f"   最大样本数: {TD3_CONFIG.get('max_samples_per_env', 500000):,}")

env_manager = EnvironmentManager(data_slices, TD3_CONFIG, device)
print("✅ 环境管理器创建成功")

✅ 使用 144 个数据切片
   每切片时长: 0.167小时
   数据采样率: 10%
   最大样本数: 5,000,000
   环境管理器使用设备: mps
   数据采样率: 10%
   最大样本数: 5,000,000
✅ 环境管理器创建成功


In [28]:
# Cell 7: TD3训练循环

def train_td3():
    """TD3主训练循环"""
    import gc
    
    # 初始化TD3算法
    td3 = TD3(
        state_dim=TD3_CONFIG['state_dim'],
        action_dim=TD3_CONFIG['action_dim'],
        action_low=TD3_CONFIG['action_low'],
        action_high=TD3_CONFIG['action_high'],
        device=device,
        lr_actor=TD3_CONFIG['lr_actor'],
        lr_critic=TD3_CONFIG['lr_critic'],
        gamma=TD3_CONFIG['gamma'],
        tau=TD3_CONFIG['tau'],
        policy_noise=TD3_CONFIG['policy_noise'],
        noise_clip=TD3_CONFIG['noise_clip'],
        policy_freq=TD3_CONFIG['policy_freq'],
        exploration_noise=TD3_CONFIG['exploration_noise']
    )
    
    # 初始化经验回放缓冲区
    replay_buffer = ReplayBuffer(TD3_CONFIG['replay_buffer_size'])
    
    # 训练统计
    episode_rewards = []
    episode_steps = []
    training_losses = []
    total_steps = 0
    best_reward = -float('inf')
    
    print(f"\n🎯 开始TD3训练")
    print(f"   设备: {device}")
    print(f"   总轮数: {TD3_CONFIG['num_episodes']:,}")
    print(f"   随机探索步数: {TD3_CONFIG['start_steps']:,}")
    print(f"   批量大小: {TD3_CONFIG['batch_size']}")
    
    # 训练循环
    for episode in tqdm(range(TD3_CONFIG['num_episodes']), desc="训练进度"):
        try:
            # 定期清理内存
            if episode % 10 == 0:
                gc.collect()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                elif device.type == 'mps':
                    torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
            
            # 获取新环境
            if episode % 100 == 0:
                print(f"\n📌 Episode {episode+1}/{TD3_CONFIG['num_episodes']}")
            
            env, data_file = env_manager.get_next_env()
            
            # 重置环境
            tensordict = env.reset()
            state = tensordict['observation'].cpu().numpy()
            
            episode_reward = 0
            episode_step = 0
            
            # Episode循环
            done = False
            while not done and episode_step < TD3_CONFIG['max_steps_per_episode']:
                # 选择动作
                if total_steps < TD3_CONFIG['start_steps']:
                    # 随机探索
                    action = np.random.uniform(
                        TD3_CONFIG['action_low'],
                        TD3_CONFIG['action_high']
                    )
                    # 确保整数参数
                    action[0] = round(action[0])
                    action[1] = round(action[1])
                    action[2] = round(action[2])
                    action[3] = round(action[3])
                else:
                    # TD3策略选择动作
                    action = td3.select_action(state, add_noise=True)
                
                # 执行动作
                action_tensor = torch.tensor(action, dtype=torch.float32, device=device)
                action_td = TensorDict({"action": action_tensor}, batch_size=(), device=device)
                
                try:
                    next_tensordict = env.step(action_td)
                except Exception as e:
                    if episode % 100 == 0:
                        print(f"   ⚠️ Step错误: {e}")
                    done = True
                    break
                
                if 'next' in next_tensordict:
                    next_state = next_tensordict['next']['observation'].cpu().numpy()
                    reward = next_tensordict['next']['reward'].cpu().item()
                    done = next_tensordict['next']['done'].cpu().item() > 0.5
                    
                    # 存储经验
                    replay_buffer.push(state, action, reward, next_state, done)
                    
                    # 更新状态
                    state = next_state
                    episode_reward += reward
                    episode_step += 1
                    total_steps += 1
                    
                    # 更新网络
                    if total_steps >= TD3_CONFIG['start_steps'] and \
                       total_steps % TD3_CONFIG['update_interval'] == 0 and \
                       len(replay_buffer) >= TD3_CONFIG['batch_size']:
                        losses = td3.update(replay_buffer, TD3_CONFIG['batch_size'])
                        if losses:
                            training_losses.append(losses)
                else:
                    done = True
            
            # 记录episode统计
            episode_rewards.append(episode_reward)
            episode_steps.append(episode_step)
            
            # 更新最佳奖励
            if episode_reward > best_reward:
                best_reward = episode_reward
                best_model_path = "checkpoints/td3/td3_model_best.pth"
                td3.save(best_model_path)
            
            # 清理环境
            env.close()
            del env
            
        except Exception as e:
            if episode % 100 == 0:
                print(f"   ❌ Episode {episode+1} 错误: {e}")
            continue
        
        # 定期评估和保存
        if (episode + 1) % TD3_CONFIG['eval_interval'] == 0:
            if episode_rewards:
                recent_rewards = episode_rewards[-min(TD3_CONFIG['eval_interval'], len(episode_rewards)):]
                avg_reward = np.mean(recent_rewards)
                avg_steps = np.mean(episode_steps[-min(TD3_CONFIG['eval_interval'], len(episode_steps)):])
                
                print(f"\n📊 Episode {episode + 1}/{TD3_CONFIG['num_episodes']}")
                print(f"   平均奖励: {avg_reward:.4f}")
                print(f"   最佳奖励: {best_reward:.4f}")
                print(f"   平均步数: {avg_steps:.0f}")
                print(f"   总步数: {total_steps:,}")
                print(f"   缓冲区大小: {len(replay_buffer):,}")
                
                # 显示最近的损失
                if training_losses and len(training_losses) > 0:
                    recent_losses = training_losses[-min(100, len(training_losses)):]
                    avg_critic_loss = np.mean([l['critic_loss'] for l in recent_losses])
                    avg_actor_loss = np.mean([l['actor_loss'] for l in recent_losses if l['actor_loss'] > 0])
                    print(f"   平均Critic损失: {avg_critic_loss:.6f}")
                    if avg_actor_loss > 0:
                        print(f"   平均Actor损失: {avg_actor_loss:.6f}")
        
        # 定期保存模型
        if (episode + 1) % TD3_CONFIG['save_interval'] == 0:
            model_path = f"checkpoints/td3/td3_model_episode_{episode + 1}.pth"
            td3.save(model_path)
            print(f"💾 模型已保存: {model_path}")
            
            # 保存训练进度
            progress = {
                'episode': episode + 1,
                'total_steps': total_steps,
                'best_reward': best_reward,
                'recent_rewards': episode_rewards[-100:] if len(episode_rewards) > 100 else episode_rewards,
            }
            progress_path = f"checkpoints/td3/training_progress.json"
            with open(progress_path, 'w') as f:
                json.dump(progress, f, indent=2)
    
    # 保存最终模型
    if episode_rewards:
        final_model_path = "checkpoints/td3/td3_model_final.pth"
        td3.save(final_model_path)
        print(f"\n✅ 训练完成！最终模型已保存: {final_model_path}")
        
        # 保存完整训练统计
        stats = {
            'episode_rewards': episode_rewards,
            'episode_steps': episode_steps,
            'total_episodes': len(episode_rewards),
            'total_steps': total_steps,
            'best_reward': best_reward,
            'config': TD3_CONFIG
        }
        
        stats_path = f"logs/td3_training_stats_{time.strftime('%Y%m%d_%H%M%S')}.json"
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f"📊 训练统计已保存: {stats_path}")
    
    return td3, episode_rewards


print("✅ TD3训练系统准备完成")
print(f"   - 使用{len(data_slices)}个数据切片轮流训练")
print(f"   - 数据采样{TD3_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   - 支持设备加速：{device}")
print(f"   - 延迟策略更新：每{TD3_CONFIG['policy_freq']}步")

✅ TD3训练系统准备完成
   - 使用144个数据切片轮流训练
   - 数据采样10%
   - 支持设备加速：mps
   - 延迟策略更新：每2步


In [29]:
# Cell 9: 执行TD3训练

print("🎯 准备开始TD3训练...")
print(f"   设备: {device}")
print(f"   数据切片: {len(data_slices)}个")

print("\n📊 当前配置（来自全局设置）：")
print(f"   训练轮数: {TD3_CONFIG['num_episodes']:,}")
print(f"   批量大小: {TD3_CONFIG['batch_size']}")
print(f"   经验池: {TD3_CONFIG['replay_buffer_size']:,}")
print(f"   数据采样率: {TD3_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   最大步数/轮: {TD3_CONFIG['max_steps_per_episode']}")
print(f"   保存间隔: 每{TD3_CONFIG['save_interval']}轮")

# 根据训练轮数显示预期
if TD3_CONFIG['num_episodes'] <= 20:
    print(f"\n🧪 快速测试模式: {TD3_CONFIG['num_episodes']}轮")
    print("   预计时间: 5-10分钟")
    print("   用途: 验证环境配置")
elif TD3_CONFIG['num_episodes'] <= 100:
    print(f"\n⚡ 快速训练模式: {TD3_CONFIG['num_episodes']}轮")
    print("   预计时间: 30-60分钟")
    print("   用途: 快速原型验证")
elif TD3_CONFIG['num_episodes'] <= 500:
    print(f"\n💪 标准训练模式: {TD3_CONFIG['num_episodes']}轮")
    print("   预计时间: 4-6小时")
    print("   用途: 获得可用策略")
else:
    print(f"\n🔥 深度训练模式: {TD3_CONFIG['num_episodes']}轮")
    print("   预计时间: 8-12小时")
    print("   用途: 获得最佳性能")
    print("   建议: 让程序在后台运行")

print("\n💡 提示: 可在第一个配置cell中调整NUM_EPISODES")

# 训练前准备
print("\n📝 训练前检查清单：")
print("   ✓ 配置参数已设置（第一个cell）")
print("   ✓ 所有前置cells已执行")
print("   ✓ 数据切片已准备")
print("   ✓ 内存充足（建议>8GB）")

try:
    print("\n✅ 环境就绪，开始训练...")
    print("-" * 50)
    
    # 清理内存
    import gc
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    elif device.type == 'mps':
        torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
    
    # 记录开始时间
    start_time = time.time()
    
    # 执行训练
    td3_agent, episode_rewards = train_td3()
    
    # 计算训练时间
    training_time = time.time() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = int(training_time % 60)
    
    # 显示结果
    print("\n🎉 训练完成！")
    print(f"   训练时间: {hours}小时 {minutes}分钟 {seconds}秒")
    
    if episode_rewards:
        print(f"\n📊 训练统计:")
        print(f"   完成轮数: {len(episode_rewards)}")
        print(f"   平均奖励: {np.mean(episode_rewards):.4f}")
        print(f"   标准差: {np.std(episode_rewards):.4f}")
        
        # 显示最后一部分的统计
        if len(episode_rewards) > 10:
            recent_n = min(50, len(episode_rewards))
            recent = episode_rewards[-recent_n:]
            print(f"\n   最后{recent_n}轮统计:")
            print(f"   平均: {np.mean(recent):.4f}")
            print(f"   最佳: {max(recent):.4f}")
            print(f"   最差: {min(recent):.4f}")
        
        # 整体统计
        print(f"\n   整体统计:")
        print(f"   最佳奖励: {max(episode_rewards):.4f}")
        print(f"   最差奖励: {min(episode_rewards):.4f}")
        success = len([r for r in episode_rewards if r > 0])
        print(f"   成功率: {success}/{len(episode_rewards)} ({success/len(episode_rewards)*100:.1f}%)")
    
    print("\n📁 已保存文件:")
    print("   - 最终模型: checkpoints/td3/td3_model_final.pth")
    print("   - 最佳模型: checkpoints/td3/td3_model_best.pth")
    print("   - 训练统计: logs/td3_training_stats_*.json")
    print("   - 训练进度: checkpoints/td3/training_progress.json")
    
    print("\n下一步：")
    print("1. 运行下一个cell进行模型评估")
    print("2. 调整训练参数（第一个配置cell）")
    print("3. 查看训练曲线和统计分析")
    
except KeyboardInterrupt:
    print("\n⚠️ 训练被用户中断")
    print("提示：")
    print("- 模型已自动保存到最近的checkpoint")
    print("- 可以从checkpoint恢复继续训练")
    
except Exception as e:
    print(f"\n❌ 错误: {e}")
    print("\n解决建议：")
    print("1. 检查内存是否充足")
    print("2. 在配置cell中减小BATCH_SIZE和REPLAY_BUFFER_SIZE")
    print("3. 在配置cell中减少DATA_SAMPLE_RATE到0.05")
    print("4. 重启kernel并重新执行")
    import traceback
    traceback.print_exc()

🎯 准备开始TD3训练...
   设备: mps
   数据切片: 144个

📊 当前配置（来自全局设置）：
   训练轮数: 200
   批量大小: 128
   经验池: 20,000
   数据采样率: 10%
   最大步数/轮: 500
   保存间隔: 每50轮

💪 标准训练模式: 200轮
   预计时间: 4-6小时
   用途: 获得可用策略

💡 提示: 可在第一个配置cell中调整NUM_EPISODES

📝 训练前检查清单：
   ✓ 配置参数已设置（第一个cell）
   ✓ 所有前置cells已执行
   ✓ 数据切片已准备
   ✓ 内存充足（建议>8GB）

✅ 环境就绪，开始训练...
--------------------------------------------------

🎯 开始TD3训练
   设备: mps
   总轮数: 200
   随机探索步数: 500
   批量大小: 128


训练进度:   0%|          | 0/200 [00:00<?, ?it/s]


📌 Episode 1/200

📊 Episode 50/200
   平均奖励: -231.1723
   最佳奖励: 353.5213
   平均步数: 167
   总步数: 8,363
   缓冲区大小: 8,363
   平均Critic损失: 2637.493643
   平均Actor损失: 43.236256
💾 模型已保存: checkpoints/td3/td3_model_episode_50.pth

📊 Episode 100/200
   平均奖励: -14.5046
   最佳奖励: 353.5213
   平均步数: 227
   总步数: 19,719
   缓冲区大小: 19,719
   平均Critic损失: 1017.360178
   平均Actor损失: 37.924034
💾 模型已保存: checkpoints/td3/td3_model_episode_100.pth

📌 Episode 101/200

📊 Episode 150/200
   平均奖励: -58.3323
   最佳奖励: 353.5213
   平均步数: 217
   总步数: 30,566
   缓冲区大小: 20,000
   平均Critic损失: 221.835213
   平均Actor损失: 18.422360
💾 模型已保存: checkpoints/td3/td3_model_episode_150.pth

📊 Episode 200/200
   平均奖励: 13.2329
   最佳奖励: 353.5213
   平均步数: 233
   总步数: 42,234
   缓冲区大小: 20,000
   平均Critic损失: 167.225489
   平均Actor损失: 20.022260
💾 模型已保存: checkpoints/td3/td3_model_episode_200.pth

✅ 训练完成！最终模型已保存: checkpoints/td3/td3_model_final.pth
📊 训练统计已保存: logs/td3_training_stats_20250808_141443.json

🎉 训练完成！
   训练时间: 0小时 11分钟 53秒

📊 训练统计:
   完成轮数: 200


In [30]:
# Cell 9: 模型评估

def evaluate_td3_agent(td3_agent, env_manager, num_eval_episodes=10):
    """评估训练好的TD3智能体"""
    
    print(f"\n🔍 评估TD3智能体性能...")
    print(f"   评估轮数: {num_eval_episodes}")
    
    eval_rewards = []
    eval_actions = []
    eval_pnls = []
    
    for episode in tqdm(range(num_eval_episodes), desc="评估进度"):
        # 获取评估环境
        env, data_file = env_manager.get_random_env()
        
        # 重置环境
        tensordict = env.reset()
        state = tensordict['observation'].cpu().numpy()
        
        episode_reward = 0
        episode_actions = []
        done = False
        steps = 0
        
        while not done and steps < TD3_CONFIG['max_steps_per_episode']:
            # 使用确定性策略（评估模式，不添加噪声）
            action = td3_agent.select_action(state, add_noise=False)
            episode_actions.append(action.tolist())
            
            # 执行动作
            action_tensor = torch.tensor(action, dtype=torch.float32, device=device)
            action_td = TensorDict({"action": action_tensor}, batch_size=(), device=device)
            next_tensordict = env.step(action_td)
            
            if 'next' in next_tensordict:
                state = next_tensordict['next']['observation'].cpu().numpy()
                reward = next_tensordict['next']['reward'].cpu().item()
                done = next_tensordict['next']['done'].cpu().item() > 0.5
                
                episode_reward += reward
                steps += 1
                
                # 记录最终PnL
                if done or steps >= TD3_CONFIG['max_steps_per_episode'] - 1:
                    strategy_state = env._get_strategy_state()
                    eval_pnls.append(strategy_state['pnl'])
            else:
                done = True
        
        eval_rewards.append(episode_reward)
        eval_actions.append(episode_actions)
        
        # 清理环境
        del env
    
    # 统计分析
    avg_reward = np.mean(eval_rewards)
    std_reward = np.std(eval_rewards)
    avg_pnl = np.mean(eval_pnls)
    success_rate = len([r for r in eval_rewards if r > 0]) / len(eval_rewards) * 100
    
    print(f"\n📊 评估结果:")
    print(f"   平均奖励: {avg_reward:.4f} ± {std_reward:.4f}")
    print(f"   平均PnL: ${avg_pnl:.2f}")
    print(f"   成功率: {success_rate:.1f}%")
    print(f"   最佳奖励: {max(eval_rewards):.4f}")
    print(f"   最差奖励: {min(eval_rewards):.4f}")
    
    # 分析学习到的策略参数
    if eval_actions:
        all_actions = [action for episode in eval_actions for action in episode]
        actions_array = np.array(all_actions)
        
        avg_params = np.mean(actions_array, axis=0)
        std_params = np.std(actions_array, axis=0)
        
        print(f"\n🎯 学习到的策略参数:")
        print(f"   {'参数':<15} {'平均值':<15} {'标准差':<10}")
        print(f"   {'-'*40}")
        
        param_names = ['半价差(ticks)', '偏度系数(ticks)', '网格层数', '网格间隔(ticks)']
        for i, name in enumerate(param_names):
            print(f"   {name:<15} {avg_params[i]:>6.1f} ± {std_params[i]:<8.1f}")
        
        print(f"\n   📌 实际使用参数（取整后）:")
        actual_params = np.round(avg_params).astype(int)
        print(f"   半价差: {actual_params[0]} ticks")
        print(f"   偏度系数: {actual_params[1]} ticks")
        print(f"   网格层数: {actual_params[2]} 层")
        print(f"   网格间隔: {actual_params[3]} ticks")
    
    return {
        'rewards': eval_rewards,
        'pnls': eval_pnls,
        'actions': eval_actions,
        'avg_reward': avg_reward,
        'avg_pnl': avg_pnl,
        'success_rate': success_rate
    }


# 如果训练完成，进行评估
if 'td3_agent' in locals():
    eval_results = evaluate_td3_agent(td3_agent, env_manager, num_eval_episodes=10)
    
    # 保存评估结果
    eval_path = f"logs/td3_eval_results_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(eval_path, 'w') as f:
        json.dump({
            'avg_reward': eval_results['avg_reward'],
            'avg_pnl': eval_results['avg_pnl'],
            'success_rate': eval_results['success_rate'],
            'rewards': eval_results['rewards'],
            'pnls': eval_results['pnls']
        }, f, indent=2)
    print(f"\n💾 评估结果已保存: {eval_path}")
else:
    print("⚠️ 请先运行训练单元格")


🔍 评估TD3智能体性能...
   评估轮数: 10


评估进度:   0%|          | 0/10 [00:00<?, ?it/s]


📊 评估结果:
   平均奖励: 112.2190 ± 115.7558
   平均PnL: $112.26
   成功率: 100.0%
   最佳奖励: 402.0144
   最差奖励: 3.4429

🎯 学习到的策略参数:
   参数              平均值             标准差       
   ----------------------------------------
   半价差(ticks)        19.5 ± 2.8     
   偏度系数(ticks)        1.7 ± 4.4     
   网格层数              10.0 ± 0.0     
   网格间隔(ticks)       19.6 ± 2.8     

   📌 实际使用参数（取整后）:
   半价差: 20 ticks
   偏度系数: 2 ticks
   网格层数: 10 层
   网格间隔: 20 ticks

💾 评估结果已保存: logs/td3_eval_results_20250808_141455.json


## 📊 总结

本 notebook 成功实现了基于 **TD3（Twin Delayed Deep Deterministic Policy Gradient）** 算法的 PMM 做市策略强化学习训练系统。

### ✅ TD3 的核心特性

1. **确定性策略**：直接输出动作，计算效率高
2. **延迟策略更新**：每2步更新一次Actor，提高稳定性
3. **目标策略平滑**：在目标动作中添加噪声，防止过拟合
4. **双Q网络**：使用较小的Q值作为目标，减少过估计

### 🎯 TD3 vs SAC 对比

| 方面 | TD3 | SAC |
|------|-----|-----|
| **策略类型** | 确定性 | 随机性 |
| **探索机制** | 高斯噪声 | 熵正则化 |
| **计算复杂度** | 较低 | 较高 |
| **样本效率** | 较高 | 中等 |
| **稳定性** | 很好 | 好 |
| **适用场景** | 动作空间较小 | 需要探索的任务 |

### 🚀 使用建议

1. **超参数调优**
   - `exploration_noise`: 控制探索程度（0.05-0.2）
   - `policy_noise`: 目标平滑噪声（0.1-0.3）
   - `policy_freq`: 策略更新频率（2-4）

2. **训练技巧**
   - 开始时使用更大的探索噪声
   - 逐渐减小噪声以精细化策略
   - 监控Actor和Critic损失的平衡

3. **性能优化**
   - TD3通常比SAC训练更快
   - 适合需要快速决策的高频交易
   - 在确定性环境中表现更好

### 💡 后续改进方向

1. **算法增强**
   - 实现 TD3+BC（Behavior Cloning）
   - 添加优先经验回放
   - 集成模仿学习

2. **策略改进**
   - 自适应噪声调节
   - 多时间尺度训练
   - 元学习快速适应

3. **实践应用**
   - 实时交易系统集成
   - 多资产组合优化
   - 风险管理增强